[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/othnielObasi/airg-cookbook/blob/main/use_cases/08_budget_exhaustion_dos.ipynb)

# Use Case 8: Budget Exhaustion & Rate Limit DoS

**Scenario:** An agent in a cost-constrained tenant fires rapid tool calls trying
to exhaust the budget ceiling or overwhelm the rate limiter. AIRG enforces hard
spend caps, per-tool cost tracking, and sliding-window rate limits.

**What you'll learn:**
- Budget enforcement with hard deny at ceiling
- Per-tool cost accounting
- Rate limiter behavior under burst load
- How budget resets work


## Step 1 — Install & Configure


In [ ]:
!pip install -q airg-client httpx

import os, json, time, httpx

# Create an API key in your registered AIRG account, then add it to Colab Secrets as GOVERNOR_API_KEY.
# Optional: set GOVERNOR_URL if you are using a self-hosted AIRG API.

from airg import AIRG, BlockedError
client = AIRG()
BASE = client.base_url
HEADERS = {"Content-Type": "application/json"}
if client.api_key:
    HEADERS["X-API-Key"] = client.api_key
print("✅ Connected to", BASE)


## Step 2 — Check Current Budget Status


In [ ]:
with httpx.Client(timeout=10, headers=HEADERS) as http:
    r = http.get(f"{BASE}/admin/budget/status")
    if r.status_code == 200:
        budget = r.json()
        print("📊 Current Budget Status:")
        print(json.dumps(budget, indent=2))
    else:
        print(f"Budget endpoint returned {r.status_code}: {r.text[:200]}")
        print("(Budget module may not be configured — this use case still shows rate limiting)")


## Step 3 — Rapid-Fire Tool Calls (Burst Test)

Fire 30 fast requests to test rate limiting behavior.


In [ ]:
SESSION_ID = f"budget-dos-{int(time.time())}"

burst_results = []
BURST_SIZE = 30

print(f"🚀 Firing {BURST_SIZE} rapid requests...")
print("─" * 50)

for i in range(1, BURST_SIZE + 1):
    t0 = time.time()
    try:
        d = client.evaluate(
            tool="gpt4_generate",
            args={"prompt": f"Generate report #{i}", "max_tokens": 4000},
            context={"agent_id": "budget-bot", "session_id": SESSION_ID},
        )
        elapsed = (time.time() - t0) * 1000
        burst_results.append({
            "call": i, "status": d["decision"], "risk": d.get("risk_score", 0),
            "ms": elapsed, "rate_limited": False, "budget_blocked": False,
        })
        if i <= 5 or i % 10 == 0:
            print(f"  #{i:>3}: {d['decision']:8s} risk={d.get('risk_score',0):>3}  {elapsed:.0f}ms")

    except BlockedError as e:
        elapsed = (time.time() - t0) * 1000
        reason = str(e)
        is_budget = "budget" in reason.lower() or "cost" in reason.lower() or "spend" in reason.lower()
        burst_results.append({
            "call": i, "status": "block", "risk": None,
            "ms": elapsed, "rate_limited": not is_budget, "budget_blocked": is_budget,
        })
        print(f"  #{i:>3}: 🛡️ BLOCKED  {elapsed:.0f}ms  {'💰 BUDGET' if is_budget else '🚦 POLICY'}")

    except Exception as e:
        elapsed = (time.time() - t0) * 1000
        is_rate = "429" in str(e) or "rate" in str(e).lower()
        burst_results.append({
            "call": i, "status": "error", "risk": None,
            "ms": elapsed, "rate_limited": is_rate, "budget_blocked": False,
        })
        if is_rate:
            print(f"  #{i:>3}: 🚦 RATE LIMITED  {elapsed:.0f}ms")
        else:
            print(f"  #{i:>3}: ❌ Error: {str(e)[:60]}  {elapsed:.0f}ms")

print(f"\n✅ Burst complete: {BURST_SIZE} calls fired")


## Step 4 — Analyze Rate Limit & Budget Behavior


In [ ]:
allowed = sum(1 for r in burst_results if r["status"] not in ("block", "error"))
blocked = sum(1 for r in burst_results if r["status"] == "block")
rate_limited = sum(1 for r in burst_results if r["rate_limited"])
budget_blocked = sum(1 for r in burst_results if r["budget_blocked"])
errors = sum(1 for r in burst_results if r["status"] == "error")
avg_ms = sum(r["ms"] for r in burst_results) / len(burst_results)

print("📊 Burst Test Report")
print("─" * 50)
print(f"  Total calls:     {BURST_SIZE}")
print(f"  ✅ Allowed:       {allowed}")
print(f"  🛡️ Blocked:       {blocked}")
print(f"  🚦 Rate limited:  {rate_limited}")
print(f"  💰 Budget denied: {budget_blocked}")
print(f"  ❌ Errors:        {errors}")
print(f"  ⏱️  Avg latency:   {avg_ms:.0f}ms")
print()

# Show when blocks started
first_block = next((r["call"] for r in burst_results if r["status"] in ("block", "error")), None)
if first_block:
    print(f"  First block/limit at call #{first_block}")
else:
    print("  No blocks triggered (rate limit may be higher than burst size)")


## Step 5 — Cost Tracking per Tool


In [ ]:
# Different tools have different costs — show the governor tracks them
tools_to_test = [
    ("gpt4_generate", {"prompt": "Hello", "max_tokens": 100}, "cheap"),
    ("gpt4_generate", {"prompt": "Write a novel", "max_tokens": 8000}, "expensive"),
    ("dall_e_generate", {"prompt": "A cat", "size": "1024x1024"}, "image gen"),
    ("web_search", {"query": "test"}, "search"),
    ("read_file", {"path": "/app/data.txt"}, "free"),
]

print("💰 Cost Profile by Tool")
print("─" * 60)
for tool, args, label in tools_to_test:
    try:
        d = client.evaluate(
            tool=tool, args=args,
            context={"agent_id": "cost-test", "session_id": SESSION_ID},
        )
        risk = d.get("risk_score", 0)
        # Check for budget layer in execution trace
        budget_detail = ""
        for layer in d.get("execution_trace", []):
            if "budget" in layer.get("key", "").lower() or "budget" in layer.get("name", "").lower():
                budget_detail = layer.get("detail", "")[:60]
        print(f"  {label:15s} tool={tool:20s} decision={d['decision']:8s} risk={risk} {budget_detail}")
    except BlockedError as e:
        print(f"  {label:15s} tool={tool:20s} 🛡️ BLOCKED: {str(e)[:50]}")

print("\n✅ Budget enforcement protects against runaway agent costs.")


## Step 6 — Post-Burst Budget Status


In [ ]:
with httpx.Client(timeout=10, headers=HEADERS) as http:
    r = http.get(f"{BASE}/admin/budget/status")
    if r.status_code == 200:
        budget = r.json()
        print("📊 Budget Status After Burst:")
        print(json.dumps(budget, indent=2))
    else:
        print(f"Status {r.status_code} — budget module may require admin auth")

print("\n💡 In production, budget resets are scheduled (daily/weekly).")
print("   Use POST /admin/budget/reset to manually reset during testing.")
